In [ ]:
import random
import os
import time

# --- 1. Configuración ---
FILAS = 6
COLUMNAS = 10
VACIO = '.'
BLOQUE = 'X'
DELAY = 0.5 # Velocidad de caída en segundos

# --- 2. Definición de las 4 Piezas ---
# Coordenadas relativas (dr, dc) de un punto de pivote (0, 0).

PIEZAS = {
    'CUADRADO': [(0, 0), (0, 1), (1, 0), (1, 1)], # O
    'BARRA':    [(0, -1), (0, 0), (0, 1), (0, 2)], # I
    'S':        [(0, 0), (0, 1), (1, -1), (1, 0)], # S
    'L':        [(0, -1), (0, 0), (0, 1), (1, 1)]  # L (derecha)
}

# --- 3. Clase Pieza ---
class Pieza:
    def __init__(self):
        nombre = random.choice(list(PIEZAS.keys()))
        self.forma = PIEZAS[nombre]
        self.r = 0  # Fila inicial (parte superior)
        self.c = COLUMNAS // 2  # Columna inicial (centro)

    def get_coords(self):
        """Devuelve las coordenadas absolutas de la pieza."""
        return [(self.r + dr, self.c + dc) for dr, dc in self.forma]

# --- 4. Lógica de Verificación ---

def es_valido(pieza_coords, fijo):
    """Verifica si las coordenadas están dentro de los límites y no colisionan con bloques fijos."""
    for r, c in pieza_coords:
        # Colisión con los límites del tablero
        if c < 0 or c >= COLUMNAS or r >= FILAS:
            return False
        # Colisión con bloques ya fijos (si r >= 0)
        if r >= 0 and (r, c) in fijo:
            return False
    return True

def dibujar_tablero(pieza_actual, fijo):
    """Crea y dibuja el tablero en la consola."""
    os.system('cls' if os.name == 'nt' else 'clear') # Limpia la consola

    # 1. Inicializar tablero vacío
    tablero = [[VACIO] * COLUMNAS for _ in range(FILAS)]

    # 2. Poner bloques fijos (Fondo)
    for r, c in fijo:
        if 0 <= r < FILAS:
            tablero[r][c] = BORDE

    # 3. Poner la pieza actual
    if pieza_actual:
        for r, c in pieza_actual.get_coords():
            if 0 <= r < FILAS and 0 <= c < COLUMNAS:
                tablero[r][c] = BLOQUE
    
    # 4. Imprimir el tablero con un borde
    borde_horizontal = '+' + ('-' * COLUMNAS) + '+'
    print(borde_horizontal)
    for fila in tablero:
        print('|' + ''.join(fila) + '|')
    print(borde_horizontal)
    print("\nControles: [A] Izquierda | [D] Derecha | [S] Abajo | [Q] Salir")

# --- 5. Bucle Principal del Juego ---

def main():
    tablero_fijo = {}  # {(r, c): BORDE}
    pieza_actual = Pieza()
    game_over = False

    while not game_over:
        dibujar_tablero(pieza_actual, tablero_fijo)

        # 1. Captura de entrada (No bloqueante)
        # Esto es un reemplazo simple para la entrada continua de consola
        movimiento = input("Mover (A/D/S) o Q: ").strip().upper()

        if movimiento == 'Q':
            break

        # 2. Intenta mover la pieza horizontal o acelerar
        nueva_pieza = Pieza()
        nueva_pieza.r, nueva_pieza.c = pieza_actual.r, pieza_actual.c

        if movimiento == 'A': # Izquierda
            nueva_pieza.c -= 1
        elif movimiento == 'D': # Derecha
            nueva_pieza.c += 1
        elif movimiento == 'S': # Abajo (acelerar la caída)
            nueva_pieza.r += 1

        if es_valido(nueva_pieza.get_coords(), tablero_fijo):
            pieza_actual = nueva_pieza
        
        # 3. Caída automática (Mover hacia abajo)
        time.sleep(DELAY) # Espera para simular el tiempo de caída

        pieza_futura = Pieza()
        pieza_futura.r, pieza_futura.c = pieza_actual.r + 1, pieza_actual.c
        pieza_futura.forma = pieza_actual.forma

        if es_valido(pieza_futura.get_coords(), tablero_fijo):
            pieza_actual.r += 1  # La pieza cae
        else:
            # 4. Fijar la pieza y generar una nueva
            for r, c in pieza_actual.get_coords():
                if r >= 0:
                    tablero_fijo[(r, c)] = BORDE

            # Limpiar líneas (simplificado, solo elimina la fila completa)
            lineas_a_limpiar = set()
            for r in range(FILAS):
                if sum(1 for c in range(COLUMNAS) if (r, c) in tablero_fijo) == COLUMNAS:
                    lineas_a_limpiar.add(r)
            
            if lineas_a_limpiar:
                # Lógica simplificada de eliminación de líneas
                nuevo_fijo = {}
                for r, c in sorted(tablero_fijo.keys()):
                    if r not in lineas_a_limpiar:
                        # Calcular cuántas líneas se eliminaron encima de esta pieza
                        desplazamiento = sum(1 for limpia_r in lineas_a_limpiar if limpia_r > r)
                        nuevo_fijo[(r + desplazamiento, c)] = BORDE
                tablero_fijo = nuevo_fijo

            # Generar nueva pieza
            pieza_actual = Pieza()
            if not es_valido(pieza_actual.get_coords(), tablero_fijo):
                game_over = True
                dibujar_tablero(None, tablero_fijo)
                print("\n*** JUEGO TERMINADO ***")
                break

if __name__ == '__main__':
    main()

+----------+
|.....XX...|
|....XX....|
|..........|
|..........|
|..........|
|..........|
+----------+

Controles: [A] Izquierda | [D] Derecha | [S] Abajo | [Q] Salir
